# Explore the Neuron Proofreader Pipeline (Demo)

An exploratory walkthrough of the `neuron-proofreader` split-correction pipeline,
one step at a time, on objects you can inspect: load fragments, generate proposals,
load ground truth, and extract features.

**This notebook is illustrative, not the training workflow.** The graphs and
proposals built here (`pred_graph`, `gt_graph`) are for inspecting counts and
tuning parameters such as `search_radius`. The actual model training and inference
rebuild their own graphs and regenerate proposals internally — they do not consume
the objects created here.

> **Related:** To actually train the model and run split correction, see
> [`run_neuron_proofreader_train_infer.ipynb`](run_neuron_proofreader_train_infer.ipynb).

**Steps:**
1. Build a `ProposalGraph` from fragment SWCs and inspect counts
2. Generate proposals and inspect how many / the effect of `search_radius`
3. Load ground-truth skeletons and (optionally) label proposals to see class balance
4. Inspect the feature-extraction pipeline

## Setup

In [ ]:
import json
import numpy as np
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../zihan_gcs_token.json"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

from neuron_proofreader.config import Config, MLConfig, ProposalGraphConfig
from neuron_proofreader.proposal_graph import ProposalGraph
from neuron_proofreader.skeleton_graph import SkeletonGraph
from neuron_proofreader.split_proofreading.split_feature_extraction import FeaturePipeline
from neuron_proofreader.split_proofreading.proposal_generation import ProposalGenerator
from neuron_proofreader.split_proofreading.groundtruth_generation import run as generate_gt_labels

## Define Paths and Configuration

Same data sources as `load_skeletons.ipynb`.

In [ ]:
# Brain and segmentation identifiers
brain_id = "794495"
segmentation_id = "raw.unet_449_splits_and_merges_900000"

# GCS paths
fragments_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/swcs"
gt_path = f"gs://allen-nd-goog/ground_truth_tracings/{brain_id}/voxel"
segmentation_path = f"gs://allen-nd-goog/from_google/{brain_id}/whole_brain/{segmentation_id}/"

# Image path (ExaSPIM raw fluorescence)
with open("../configs/exaspim_image_prefixes.json") as f:
    img_prefixes = json.load(f)
img_path = img_prefixes[brain_id]

print(f"Fragments: {fragments_path}")
print(f"Ground truth: {gt_path}")
print(f"Segmentation: {segmentation_path}")
print(f"Image: {img_path}")

In [ ]:
# Configuration for the proofreading pipeline.
# These parameters control graph construction and proposal generation.

graph_config = ProposalGraphConfig(
    anisotropy=(0.748, 0.748, 1.0),   # ExaSPIM voxel size in microns (x, y, z)
    min_cable_length=40.0,             # Drop fragments shorter than this (microns)
    node_spacing=1.0,                  # Resample spacing between nodes (microns)
    prune_depth=24.0,                  # Prune short terminal branches (microns)
    max_proposals_per_leaf=3,          # Max connection candidates per leaf node
    allow_nonleaf_proposals=False,     # Only propose from leaf (endpoint) nodes
    remove_high_risk_merges=False,
    verbose=True,
)

ml_config = MLConfig(
    batch_size=64,
    brightness_clip=400,               # Clip image intensity for normalization
    device="cuda",                     # Use GPU for inference
    patch_shape=(96, 96, 96),           # 3D image patch around each proposal
    transform=False,                   # No augmentation at inference time
)

config = Config(graph_config, ml_config)
print("Graph config:", graph_config)
print("ML config:", ml_config)

## Section 1: Build the ProposalGraph

Load fragment skeletons from SWC files into a `ProposalGraph`. This is the same
data as `dataset.fragments_graph` in `load_skeletons_from_cache.ipynb`, but
loaded through `neuron-proofreader`'s own graph class which adds proposal tracking.

*Illustrative — the train/inference notebook rebuilds this internally and does not
use `pred_graph`.*

In [ ]:
# Build the ProposalGraph from fragment SWC files
pred_graph = ProposalGraph(
    anisotropy=graph_config.anisotropy,
    max_proposals_per_leaf=graph_config.max_proposals_per_leaf,
    min_cable_length=graph_config.min_cable_length,
    node_spacing=graph_config.node_spacing,
    prune_depth=graph_config.prune_depth,
    remove_high_risk_merges=graph_config.remove_high_risk_merges,
    verbose=graph_config.verbose,
)

pred_graph.load(fragments_path)

print(f"Fragments loaded:")
print(f"  Nodes: {pred_graph.number_of_nodes():,}")
print(f"  Edges: {pred_graph.number_of_edges():,}")
print(f"  Components: {len(set(pred_graph.node_component_id)):,}")

## Section 2: Generate Proposals

For each leaf node (fragment endpoint), search for nearby nodes from *other*
fragments. Each nearby pair becomes a "proposal" — a candidate reconnection
that the model will score as accept/reject.

The `search_radius` controls how far to look (in microns). Larger radius =
more proposals but slower. Typical values: 10-30 microns.

*Illustrative — training and inference regenerate proposals on their own graphs
with the same `search_radius`.*

In [ ]:
search_radius = 20.0  # microns

pred_graph.generate_proposals(
    search_radius=search_radius,
    allow_nonleaf_proposals=graph_config.allow_nonleaf_proposals,
)

n_proposals = pred_graph.n_proposals()
print(f"Proposals generated: {n_proposals:,}")
print(f"Search radius: {search_radius} microns")
print(f"Max proposals per leaf: {graph_config.max_proposals_per_leaf}")

## Section 3: Generate Ground-Truth Labels (for training)

If ground-truth skeletons are available, we can label each proposal as
accept (both endpoints align to the same GT neuron) or reject. This is
used to train the GNN model.

A proposal is accepted (see `groundtruth_generation.run`) if:
- Both fragments align to the same GT skeleton (alignment scored per connected
  component in `find_aligned_component`)
- The proposal's projection distance onto the GT skeleton is small
  (90th-percentile distance < 8 microns)
- The connection is structurally consistent with GT topology — it joins the same
  or adjacent GT edges, and for adjacent edges the proposal length matches the GT
  path length to within 40 microns

In [ ]:
# Load ground-truth skeletons
gt_graph = SkeletonGraph(
    anisotropy=graph_config.anisotropy,
    min_cable_length=0,
    node_spacing=graph_config.node_spacing,
    verbose=True,
)
gt_graph.load(gt_path)

print(f"Ground truth loaded:")
print(f"  Nodes: {gt_graph.number_of_nodes():,}")
print(f"  Edges: {gt_graph.number_of_edges():,}")
print(f"  Components (neurons): {len(set(gt_graph.node_component_id)):,}")

In [ ]:
# # Generate accept/reject labels for each proposal (shows training class balance)
# gt_accepts = generate_gt_labels(gt_graph, pred_graph)

# n_accepts = len(gt_accepts)
# n_rejects = n_proposals - n_accepts

# print(f"Ground-truth labels:")
# print(f"  Accept: {n_accepts:,} ({100*n_accepts/max(n_proposals,1):.1f}%)")
# print(f"  Reject: {n_rejects:,} ({100*n_rejects/max(n_proposals,1):.1f}%)")

## Section 4: Feature Extraction

For each proposal, the `FeaturePipeline` runs two extractors:
- **Skeleton features** (`SkeletonFeatureExtractor`): per-node (degree, radius,
  number of proposals), per-edge (mean radius, normalized path length), and
  per-proposal (leaf-to-leaf flag, nearby leaf counts, normalized length, radius,
  multi-scale directionals) geometry
- **Image features** (`ImageFeatureExtractor`): a 3D image patch (96x96x96)
  centered on each proposal, extracted from the raw fluorescence volume

These are packed into a `HeteroGraphData` object (PyTorch Geometric) for the GNN.

In [ ]:
# # Initialize the feature extraction pipeline
# feature_pipeline = FeaturePipeline(
#     graph=pred_graph,
#     img_path=img_path,
#     brightness_clip=ml_config.brightness_clip,
#     patch_shape=ml_config.patch_shape,
# )

# print(f"FeaturePipeline initialized")
# print(f"  Image path: {img_path}")
# print(f"  Patch shape: {ml_config.patch_shape}")
# print(f"  Brightness clip: {ml_config.brightness_clip}")

## Alternative: Use Cached BrainDataset

If you already have a `BrainDataset` cached from `load_skeletons_from_cache.ipynb`,
you can reuse its file paths directly instead of re-specifying them.

In [ ]:
from agentic_neuron_proofreader.data_modules.datasets import BrainDataset

# Load the cached dataset (fast — no SWC re-reading)
cache_path = f"../dataset_cache_{brain_id}.pkl"
if os.path.exists(cache_path):
    dataset = BrainDataset.load_from_cache(cache_path)

    # The cached dataset holds the same data that neuron-proofreader needs.
    # Key attributes you can reuse:
    print("Cached BrainDataset loaded")
    print(f"  fragments_graph nodes: {dataset.fragments_graph.number_of_nodes():,}")
    print(f"  gt_graph nodes: {dataset.gt_graph.number_of_nodes():,}")
    print(f"  img_path: {dataset.img_path}")
    print()
    print("To use with neuron-proofreader, pass the same file paths:")
    print(f"  fragments_path = '{fragments_path}'")
    print(f"  gt_path = '{gt_path}'")
    print(f"  img_path = '{dataset.img_path}'")
else:
    print(f"Cache not found: {cache_path}")
    print("Run load_skeletons.ipynb first to generate the cache.")